In [1]:
# =========================================
# 07 - Inference Demo for Hybrid Stacking Model + Big Five + MBTI
# =========================================

import os
import re
import json
import numpy as np
import pandas as pd
import joblib
import torch
from transformers import AutoTokenizer, AutoModel

# اگر پروژه را در مسیر دیگری گذاشتی، فقط همین متغیر را دستی تغییر بده.
# حالت پیش‌فرض: پیدا کردن مسیر اصلی پروژه از محل همین نوت‌بوک یا current working directory.
def find_project_root(start=None):
    start = os.path.abspath(start or os.getcwd())
    candidates = [start, os.path.dirname(start), os.path.dirname(os.path.dirname(start))]
    for c in candidates:
        if os.path.exists(os.path.join(c, "data")) and os.path.exists(os.path.join(c, "notebooks")):
            return c
    # مسیر قبلی ویندوزی شما به عنوان fallback
    return r"C:\Users\Neda\Desktop\personality_llm"

BASE_PATH = find_project_root()

FEATURES_PATH = os.path.join(BASE_PATH, "data", "features")
MODELS_PATH = os.path.join(BASE_PATH, "models")

HYBRID_MODEL_PATH = os.path.join(MODELS_PATH, "hybrid_stacking_model.pkl")
if not os.path.exists(HYBRID_MODEL_PATH):
    HYBRID_MODEL_PATH = os.path.join(MODELS_PATH, "hybrid_model.pkl")

LABEL_ENCODER_PATH = os.path.join(MODELS_PATH, "label_encoder.pkl")
SCHEMA_PATH = os.path.join(MODELS_PATH, "hybrid_feature_schema.json")
BERT_PATH = os.path.join(MODELS_PATH, "parsbert_model")

print("BASE_PATH:", BASE_PATH)
print("Model path:", HYBRID_MODEL_PATH)



# =========================================
# 1. LOAD MODEL, LABEL ENCODER, SCHEMA
# =========================================

model = joblib.load(HYBRID_MODEL_PATH)
label_encoder = joblib.load(LABEL_ENCODER_PATH)

with open(SCHEMA_PATH, "r", encoding="utf-8") as f:
    schema = json.load(f)

STYLE_COLUMNS = schema["style_columns"]
BERT_DIM = int(schema["bert_dim"])
USE_LLM_FEATURES = bool(schema.get("use_llm_features", False))
LLM_FEATURE_COLUMNS = schema.get("llm_feature_columns", [])
TOTAL_DIM = int(schema["total_dim"])

print("Number of style columns:", len(STYLE_COLUMNS))
print("BERT dim:", BERT_DIM)
print("Use LLM features:", USE_LLM_FEATURES)
print("Total expected features:", TOTAL_DIM)
print("Model expected features:", model.n_features_in_)

assert TOTAL_DIM == model.n_features_in_, "Schema and model feature dimensions do not match."



# =========================================
# 2. LOAD PARSBERT
# =========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(BERT_PATH)
bert_model = AutoModel.from_pretrained(BERT_PATH)
bert_model.to(device)
bert_model.eval()

print("✔ Models loaded")



# =========================================
# 3. TEXT CLEANING
# =========================================

def clean_text(text):
    text = str(text)
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = text.replace("ة", "ه").replace("ۀ", "ه")
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = text.replace("#", " ")
    text = re.sub(r"[A-Za-z]", " ", text)
    text = re.sub(r"[0-9۰-۹]", " ", text)
    text = re.sub(r"[^\u0600-\u06FF\s\.\!\؟\?،؛]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text



# =========================================
# 4. STYLE FEATURE EXTRACTION
# =========================================

POS_WORDS = {"خوب", "عالی", "دوست", "موفق", "زیبا", "خوشحال", "امید", "مثبت", "رضایت", "حمایت"}
NEG_WORDS = {"بد", "ضعیف", "مشکل", "ناراحت", "نگران", "ترس", "استرس", "منفی", "اشتباه", "بحران"}
FIRST_PERSON = {"من", "خودم", "ما", "برایم", "برام", "به‌نظرم", "فکر", "می‌کنم"}
SOCIAL = {"دوست", "خانواده", "همکار", "گروه", "مردم", "دیگران", "کمک", "حمایت", "تیم"}
CERTAINTY = {"حتما", "قطعاً", "مطمئن", "یقیناً", "بدون شک"}
UNCERTAINTY = {"شاید", "احتمالاً", "ممکن", "نمی‌دانم", "فکر کنم", "ظاهراً"}

def count_words(text, word_set):
    return sum(1 for w in word_set if w in text)

def lexical_diversity(text):
    words = str(text).split()
    return len(set(words)) / len(words) if words else 0

def sentence_count(text):
    return len([s for s in re.split(r"[.!؟?]+", str(text)) if s.strip()])

def avg_sentence_length(text):
    sentences = [s for s in re.split(r"[.!؟?]+", str(text)) if s.strip()]
    return np.mean([len(s.split()) for s in sentences]) if sentences else 0

def repeated_chars(text):
    return len(re.findall(r"(.)\1{2,}", str(text)))

def extract_style_features(text):
    text = clean_text(text)
    words = text.split()
    n_words = len(words) if len(words) > 0 else 1

    features = {
        "num_words": len(words),
        "num_chars": len(text),
        "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
        "type_token_ratio": len(set(words)) / len(words) if words else 0,
        "num_questions": text.count("?") + text.count("؟"),
        "num_exclamations": text.count("!"),
        "num_commas": text.count("،"),
        "num_sentences": sentence_count(text),
        "lexical_diversity": lexical_diversity(text),
        "sentence_count": sentence_count(text),
        "avg_sentence_length": avg_sentence_length(text),
        "repeated_char_count": repeated_chars(text),
        "pos_count": count_words(text, POS_WORDS),
        "neg_count": count_words(text, NEG_WORDS),
        "first_person": count_words(text, FIRST_PERSON),
        "social_words": count_words(text, SOCIAL),
        "certainty": count_words(text, CERTAINTY),
        "uncertainty": count_words(text, UNCERTAINTY),
        "pos_ratio": count_words(text, POS_WORDS) / n_words,
        "neg_ratio": count_words(text, NEG_WORDS) / n_words,
        "first_person_ratio": count_words(text, FIRST_PERSON) / n_words,
        "social_ratio": count_words(text, SOCIAL) / n_words,
        "certainty_ratio": count_words(text, CERTAINTY) / n_words,
        "uncertainty_ratio": count_words(text, UNCERTAINTY) / n_words,
    }

    style_vector = pd.DataFrame([[features.get(col, 0) for col in STYLE_COLUMNS]], columns=STYLE_COLUMNS)
    return style_vector.values.astype(float)



# =========================================
# 5. BERT EMBEDDING
# =========================================

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def get_bert_embedding(text):
    text = clean_text(text)
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = bert_model(**encoded)
        embedding = mean_pooling(outputs.last_hidden_state, encoded["attention_mask"])

    emb = embedding.cpu().numpy()
    if emb.shape[1] != BERT_DIM:
        raise ValueError(f"BERT dim mismatch: {emb.shape[1]} vs {BERT_DIM}")
    return emb


# =========================================
# 6. LLM-LIKE BIG FIVE FEATURES + MBTI HELPERS FOR DEMO
# =========================================
# نکته پایان‌نامه‌ای:
# این بخش برای Demo است. اگر خروجی واقعی LLM داری، llm_scores را دستی یا از API بده.
# اگر llm_scores ندهی، یک برآورد ساده و قابل توضیح از روی نشانه‌های زبانی ساخته می‌شود.

TRAITS = ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]

TRAIT_FA = {
    "openness": "گشودگی به تجربه / Openness",
    "conscientiousness": "وظیفه‌شناسی / Conscientiousness",
    "extraversion": "برون‌گرایی / Extraversion",
    "agreeableness": "سازگاری / Agreeableness",
    "neuroticism": "روان‌رنجوری / Neuroticism",
}

TRAIT_DEFAULTS = {
    "openness": 3,
    "conscientiousness": 3,
    "extraversion": 3,
    "agreeableness": 3,
    "neuroticism": 3,
    "llm_confidence": 0.50,
}

OPENNESS_WORDS = {
    "تجربه", "جدید", "خلاق", "ایده", "یادگیری", "هنر", "کنجکاو", "کشف", "تغییر", "متفاوت", "ماجراجویی"
}
CONSCIENTIOUS_WORDS = {
    "دقیق", "منظم", "برنامه", "هدف", "مسئولیت", "پیگیری", "زمان", "تعهد", "کامل", "وظیفه", "ساختار"
}
EXTRAVERSION_WORDS = {
    "جمع", "مهمانی", "اجتماعی", "دوستان", "صحبت", "گفتگو", "انرژی", "گروه", "ارتباط", "هیجان"
}
AGREEABLENESS_WORDS = {
    "کمک", "حمایت", "مهربان", "همدلی", "درک", "بخشش", "همکاری", "احترام", "دیگران", "دوست"
}
NEUROTICISM_WORDS = {
    "نگران", "استرس", "اضطراب", "ترس", "ناراحت", "آینده", "فشار", "غم", "دلشوره", "ذهنم", "مشکل"
}

LOW_EXTRAVERSION_WORDS = {"تنهایی", "خلوت", "ساکت", "آرام", "تنها"}
LOW_NEUROTICISM_WORDS = {"آرام", "مطمئن", "کنترل", "راحت", "ثبات"}

def _score_from_keywords(text, positive_words, negative_words=None, base=3):
    text = clean_text(text)
    pos = sum(1 for w in positive_words if w in text)
    neg = sum(1 for w in (negative_words or set()) if w in text)
    score = base + min(pos, 2) - min(neg, 1)
    return int(np.clip(score, 1, 5))

def estimate_bigfive_from_text(text):
    """
    Demo estimator: امتیازهای 1 تا 5 برای Big Five می‌سازد.
    برای کار نهایی می‌توانی این تابع را با خروجی واقعی LLM جایگزین کنی.
    """
    text_clean = clean_text(text)
    words = text_clean.split()
    n_words = max(len(words), 1)

    scores = {
        "openness": _score_from_keywords(text_clean, OPENNESS_WORDS),
        "conscientiousness": _score_from_keywords(text_clean, CONSCIENTIOUS_WORDS),
        "extraversion": _score_from_keywords(text_clean, EXTRAVERSION_WORDS, LOW_EXTRAVERSION_WORDS),
        "agreeableness": _score_from_keywords(text_clean, AGREEABLENESS_WORDS),
        "neuroticism": _score_from_keywords(text_clean, NEUROTICISM_WORDS, LOW_NEUROTICISM_WORDS),
    }

    # چند قانون سبک نوشتاری ساده برای طبیعی‌تر شدن Demo
    if text_clean.count("!") >= 1:
        scores["extraversion"] = int(np.clip(scores["extraversion"] + 1, 1, 5))
    if text_clean.count("؟") + text_clean.count("?") >= 1:
        scores["openness"] = int(np.clip(scores["openness"] + 1, 1, 5))
    if any(w in text_clean for w in ["باید", "حتما", "قطعاً", "تصمیم", "بررسی"]):
        scores["conscientiousness"] = int(np.clip(scores["conscientiousness"] + 1, 1, 5))
    if sum(1 for w in FIRST_PERSON if w in text_clean) / n_words > 0.10:
        scores["neuroticism"] = int(np.clip(scores["neuroticism"] + 1, 1, 5))

    scores["llm_confidence"] = 0.55
    return scores

def level_fa(score):
    if score >= 4:
        return "بالا"
    if score <= 2:
        return "پایین"
    return "متوسط"

def bigfive_to_mbti(bigfive_scores):
    """
    نگاشت تقریبی و نمایشی Big Five به MBTI:
    E/I ← Extraversion
    N/S ← Openness
    F/T ← Agreeableness
    J/P ← Conscientiousness
    Neuroticism در MBTI مستقیم وارد نمی‌شود و فقط در توضیح می‌آید.
    """
    e_i = "E" if bigfive_scores["extraversion"] >= 4 else "I"
    n_s = "N" if bigfive_scores["openness"] >= 4 else "S"
    f_t = "F" if bigfive_scores["agreeableness"] >= 4 else "T"
    j_p = "J" if bigfive_scores["conscientiousness"] >= 4 else "P"
    return e_i + n_s + f_t + j_p

def get_llm_features_for_text(text, llm_scores=None):
    if not USE_LLM_FEATURES:
        return np.empty((1, 0))

    # قبلاً neutral بود؛ الان برای Demo از برآورد Big Five استفاده می‌کنیم.
    if llm_scores is None:
        llm_scores = estimate_bigfive_from_text(text)

    values = []
    for col in LLM_FEATURE_COLUMNS:
        values.append(float(llm_scores.get(col, TRAIT_DEFAULTS.get(col, 0))))

    return np.array([values], dtype=float)


# =========================================
# 7. PREDICT FUNCTION: MODEL LABEL + BIG FIVE + MBTI
# =========================================

def predict_personality(text, llm_scores=None, return_proba=True):
    # برای نمایش Big Five/MBTI
    display_bigfive = llm_scores.copy() if llm_scores is not None else estimate_bigfive_from_text(text)
    for t in TRAITS:
        display_bigfive[t] = int(np.clip(round(float(display_bigfive.get(t, 3))), 1, 5))
    display_bigfive["llm_confidence"] = float(display_bigfive.get("llm_confidence", 0.55))

    # برای مدل hybrid
    style_vec = extract_style_features(text)
    bert_vec = get_bert_embedding(text)
    llm_vec = get_llm_features_for_text(text, llm_scores=display_bigfive)

    x = np.hstack([style_vec, bert_vec, llm_vec])

    if x.shape[1] != model.n_features_in_:
        raise ValueError(f"Feature mismatch: {x.shape[1]} vs {model.n_features_in_}")

    pred_encoded = model.predict(x)[0]
    pred_label = label_encoder.inverse_transform([pred_encoded])[0]

    mbti = bigfive_to_mbti(display_bigfive)

    result = {
        "text": text,
        "predicted_dataset_label": pred_label,
        "big_five_scores": {t: display_bigfive[t] for t in TRAITS},
        "big_five_levels": {t: level_fa(display_bigfive[t]) for t in TRAITS},
        "mbti_estimate": mbti,
        "confidence": display_bigfive.get("llm_confidence", 0.55),
        "note": "MBTI فقط تفسیر تقریبی از Big Five است و خروجی اصلی/علمی مدل محسوب نمی‌شود."
    }

    if return_proba and hasattr(model, "predict_proba"):
        proba = model.predict_proba(x)[0]
        result["dataset_label_probabilities"] = {
            str(label_encoder.inverse_transform([i])[0]): float(proba[i])
            for i in range(len(proba))
        }

    return result

def print_demo_result(result):
    print("\n" + "="*70)
    print("متن ورودی:")
    print(result["text"])
    print("\nخروجی مدل دیتاست:")
    print("Predicted Label:", result["predicted_dataset_label"])
    print("\nخروجی Big Five:")
    for trait, score in result["big_five_scores"].items():
        print(f"- {TRAIT_FA[trait]}: {score}/5 ({result['big_five_levels'][trait]})")
    print("\nتفسیر MBTI تقریبی:")
    print(result["mbti_estimate"])
    print("\nتذکر:")
    print(result["note"])


# =========================================
# 8. DEMO: خروجی مورد نظر پایان‌نامه/ارائه
# =========================================

texts = [
    "من عاشق تجربه‌های جدیدم، ولی قبل از تصمیم‌گیری زیاد فکر می‌کنم و گاهی نگران آینده می‌شوم.",
    "من کارها را دقیق، منظم و طبق برنامه انجام می‌دهم و دوست دارم همه چیز کامل باشد.",
    "من در جمع انرژی می‌گیرم، با آدم‌ها سریع ارتباط می‌گیرم و از گفتگو با دیگران لذت می‌برم.",
]

print("\n================ DEMO: Big Five + MBTI ================")

for text in texts:
    result = predict_personality(text, return_proba=False)
    print_demo_result(result)


# =========================================
# 9. EXAMPLE WITH MANUAL/REAL LLM SCORES
# =========================================
# وقتی با ChatGPT، Gemini، Llama، Qwen یا هر LLM دیگری امتیاز واقعی گرفتی، اینجا بده.
# این روش برای دفاع بهتر از heuristic است.

example_text = "من از صحبت کردن با آدم‌های جدید انرژی می‌گیرم و معمولاً توی جمع‌ها راحت ارتباط برقرار می‌کنم. دوست دارم تجربه‌های جدید امتحان کنم حتی اگر ریسک داشته باشند. خیلی وقت‌ها بدون برنامه‌ریزی دقیق تصمیم می‌گیرم و بعدش مسیرم رو اصلاح می‌کنم. ولی در عین حال وقتی کسی به من نیاز داشته باشه سعی می‌کنم کمکش کنم"
example_llm_scores = {
    "openness": 5,
    "conscientiousness": 4,
    "extraversion": 2,
    "agreeableness": 4,
    "neuroticism": 4,
    "llm_confidence": 0.82,
}

result = predict_personality(example_text, llm_scores=example_llm_scores, return_proba=False)
print_demo_result(result)




BASE_PATH: C:\Users\Neda\Desktop\personality_llm
Model path: C:\Users\Neda\Desktop\personality_llm\models\hybrid_stacking_model.pkl
Number of style columns: 24
BERT dim: 768
Use LLM features: False
Total expected features: 792
Model expected features: 792


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: C:\Users\Neda\Desktop\personality_llm\models\parsbert_model
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✔ Models loaded

================ DEMO: Big Five + MBTI ================

متن ورودی:
من عاشق تجربه‌های جدیدم، ولی قبل از تصمیم‌گیری زیاد فکر می‌کنم و گاهی نگران آینده می‌شوم.

خروجی مدل دیتاست:
Predicted Label: 6

خروجی Big Five:
- گشودگی به تجربه / Openness: 5/5 (بالا)
- وظیفه‌شناسی / Conscientiousness: 4/5 (بالا)
- برون‌گرایی / Extraversion: 3/5 (متوسط)
- سازگاری / Agreeableness: 3/5 (متوسط)
- روان‌رنجوری / Neuroticism: 5/5 (بالا)

تفسیر MBTI تقریبی:
INTJ

تذکر:
MBTI فقط تفسیر تقریبی از Big Five است و خروجی اصلی/علمی مدل محسوب نمی‌شود.

متن ورودی:
من کارها را دقیق، منظم و طبق برنامه انجام می‌دهم و دوست دارم همه چیز کامل باشد.

خروجی مدل دیتاست:
Predicted Label: 3

خروجی Big Five:
- گشودگی به تجربه / Openness: 3/5 (متوسط)
- وظیفه‌شناسی / Conscientiousness: 5/5 (بالا)
- برون‌گرایی / Extraversion: 3/5 (متوسط)
- سازگاری / Agreeableness: 4/5 (بالا)
- روان‌رنجوری / Neuroticism: 3/5 (متوسط)

تفسیر MBTI تقریبی:
ISFJ

تذکر:
MBTI فقط تفسیر تقریبی از Big Five است و خروجی اصلی/علمی مدل محسوب نمی